In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [5]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

In [6]:
student_df = pd.read_csv('/content/gdrive/MyDrive/mental_health_prdiction/cleaned_student_dataset.csv')
reddit_df = pd.read_csv('/content/gdrive/MyDrive/mental_health_prdiction/cleaned_reddit_dataset.csv')

In [7]:
student_df.head()

,Gender,Age,Academic Pressure,Work Pressure,CGPA,Sleep Duration,Dietary Habits,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression,serialized_text
0,Male,33,5.0,0.0,4.49,'5-6 hours',Healthy,3.0,1.0,0,1,"A 33-year-old Male student with CGPA 4.49, 5.0..."
1,Female,24,2.0,0.0,2.95,'5-6 hours',Moderate,3.0,2.0,1,0,"A 24-year-old Female student with CGPA 2.95, 2..."
2,Male,31,3.0,0.0,3.52,'Less than 5 hours',Healthy,9.0,1.0,1,0,"A 31-year-old Male student with CGPA 3.52, 3.0..."
3,Female,28,3.0,0.0,2.80,'7-8 hours',Moderate,4.0,5.0,1,1,"A 28-year-old Female student with CGPA 2.8, 3...."
4,Female,25,4.0,0.0,4.07,'5-6 hours',Moderate,1.0,1.0,0,0,"A 25-year-old Female student with CGPA 4.07, 4..."


In [8]:
student_df = student_df.drop(['serialized_text'], axis=1)

In [9]:
def serialize_student_row(row):
    """
    Converts your specific columns into a clinical text sentence.
    """
    # Handling floats by converting to int first
    pressure_map = {0: 'no', 1: 'very low', 2: 'low', 3: 'moderate', 4: 'high', 5: 'extreme'}

    # int(float(...)) to handle '5.0' appearing as a string or float
    try:
        academic_pressure = pressure_map.get(int(float(row.get('Academic Pressure', 3))), 'moderate')
        work_pressure = pressure_map.get(int(float(row.get('Work Pressure', 0))), 'low')
        financial_stress  = pressure_map.get(int(float(row.get('Financial Stress', 0))), 'low')
        family_history = 'Yes' if int(float(row.get('Family History of Mental Illness', 0))) == 1 else 'No'
    except:

        academic_pressure, work_pressure, financial_stress, family_history = 'moderate', 'low', 'low', 'No'

    # sentence Build
    text = (
        f"[CLINICAL] Subject is a {int(float(row.get('Age', 20)))}-year-old {row.get('Gender', 'Student')}. "
        f"Academic standing: CGPA of {row.get('CGPA', 0.0)}. "
        f"They report {academic_pressure} academic pressure and {work_pressure} work pressure. "
        f"Sleep duration: {row.get('Sleep Duration', 'unknown')}. "
        f"Dietary habits: {row.get('Dietary Habits', 'unknown')}. "
        f"Financial stress is {financial_stress}. "
        f"Family history of mental illness: {family_history}."
    )
    return text

In [10]:
# translation
student_df['text'] = student_df.apply(serialize_student_row, axis=1)

In [11]:
print("Student Depression labels:")
print(student_df['Depression'].value_counts(dropna=False))

print("\nReddit binary labels:")
print(reddit_df['binary_label'].value_counts(dropna=False))

Student Depression labels:
Depression
1    16336
0    11565
Name: count, dtype: int64

Reddit binary labels:
binary_label
1    26056
0    16343
Name: count, dtype: int64


In [12]:
print("Student Depression labels:")
print(student_df['Depression'].value_counts(dropna=False))

print("\nReddit binary labels:")
print(reddit_df['binary_label'].value_counts(dropna=False))

Student Depression labels:
Depression
1    16336
0    11565
Name: count, dtype: int64

Reddit binary labels:
binary_label
1    26056
0    16343
Name: count, dtype: int64


In [13]:
print(student_df[['Depression', 'text']].sample(10, random_state=42).to_string(index=False))

 Depression                                                                                                                                                                                                                                                                        text
          0       [CLINICAL] Subject is a 29-year-old Female. Academic standing: CGPA of 4.26. They report low academic pressure and no work pressure. Sleep duration: 'More than 8 hours'. Dietary habits: Healthy. Financial stress is extreme. Family history of mental illness: No.
          0             [CLINICAL] Subject is a 28-year-old Male. Academic standing: CGPA of 2.78. They report low academic pressure and no work pressure. Sleep duration: '7-8 hours'. Dietary habits: Unhealthy. Financial stress is very low. Family history of mental illness: Yes.
          0 [CLINICAL] Subject is a 34-year-old Male. Academic standing: CGPA of 2.56. They report moderate academic pressure and no work pressure. Sleep durati

In [14]:
print(reddit_df[['binary_label', 'clean_text']].sample(10, random_state=42).to_string(index=False))

 binary_label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              clean_text
            1 tyson, fury and countless other champions had to get through it why would it be any easier for me, this d actually hurted me so much

In [15]:
print("Student Depression labels:")
print(student_df['Depression'].value_counts(dropna=False))

print("\nReddit binary labels:")
print(reddit_df['binary_label'].value_counts(dropna=False))

Student Depression labels:
Depression
1    16336
0    11565
Name: count, dtype: int64

Reddit binary labels:
binary_label
1    26056
0    16343
Name: count, dtype: int64


In [17]:
# FUSING DATASETS
# Prepare Student Subset (Text + Label)
student_subset = student_df[['text', 'Depression']].rename(columns={'Depression': 'labels'})
# Prepare Reddit Subset (Text + Label)
reddit_subset = reddit_df[['clean_text', 'binary_label']].rename(columns={'clean_textt': 'text', 'binary_label': 'labels'})

# Vertical Stack
full_dataset = pd.concat([student_subset, reddit_subset], axis=0)

# Shuffle & Split
full_dataset = full_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
train_df, test_df = train_test_split(full_dataset, test_size=0.2, random_state=42, stratify=full_dataset['labels'])

In [18]:
print("Columns:", full_dataset.columns.tolist())

print("\nDataset shape:", full_dataset.shape)

print("\nMissing values:")
print(full_dataset.isna().sum())

print("\nSample records:")
display(full_dataset.head())

Columns: ['text', 'labels', 'clean_text']

Dataset shape: (70300, 3)

Missing values:
text          42399
labels            0
clean_text    27902
dtype: int64

Sample records:


,text,labels,clean_text
0,NaN,1,do not expect me to do anything. do not expect...
1,[CLINICAL] Subject is a 20-year-old Female. Ac...,1,NaN
2,[CLINICAL] Subject is a 22-year-old Female. Ac...,0,NaN
3,NaN,1,i have not been sleeping for 8 hours a day for...
4,[CLINICAL] Subject is a 21-year-old Female. Ac...,1,NaN


In [19]:
print("Training text missing:", train_df["text"].isna().sum())
print("Testing text missing:", test_df["text"].isna().sum())

print("\nTraining text examples:")
print(train_df["text"].head(10).tolist())

print("\nTraining label distribution:")
print(train_df["labels"].value_counts())

Training text missing: 33928
Testing text missing: 8471

Training text examples:
[nan, "[CLINICAL] Subject is a 29-year-old Male. Academic standing: CGPA of 4.8. They report moderate academic pressure and no work pressure. Sleep duration: '5-6 hours'. Dietary habits: Unhealthy. Financial stress is very low. Family history of mental illness: No.", "[CLINICAL] Subject is a 28-year-old Male. Academic standing: CGPA of 4.52. They report moderate academic pressure and no work pressure. Sleep duration: '5-6 hours'. Dietary habits: Unhealthy. Financial stress is moderate. Family history of mental illness: No.", nan, "[CLINICAL] Subject is a 20-year-old Female. Academic standing: CGPA of 4.02. They report low academic pressure and no work pressure. Sleep duration: 'More than 8 hours'. Dietary habits: Moderate. Financial stress is high. Family history of mental illness: No.", nan, nan, nan, nan, nan]

Training label distribution:
labels
1    33914
0    22326
Name: count, dtype: int64


In [21]:
#Missing Values Instead of Dropping and Fill NaNs with empty strings
reddit_subset['clean_text'] = reddit_subset['clean_text'].fillna("Empty Post")
student_subset['text'] = student_subset['text'].fillna("Empty Note")

In [22]:
# CLEANING(some Reddit dataset has some empty rows or numbers mixed in with the posts)

# we Convert everything to String
train_df['text'] = train_df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)

# we Remove any rows that are still empty or just "nan"
train_df = train_df[train_df['text'].str.strip() != 'nan']
test_df = test_df[test_df['text'].str.strip() != 'nan']
train_df = train_df[train_df['text'].notna()]
test_df = test_df[test_df['text'].notna()]

In [ ]:
from huggingface_hub import login
login()

In [ ]:
#TOKENIZATION
model_id = "mental/mental-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

hf_train = Dataset.from_pandas(train_df)
hf_test = Dataset.from_pandas(test_df)

tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_test = hf_test.map(tokenize_function, batched=True)

In [ ]:
# TRAINING
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Weight class 1 (Depressed) higher to ensure we catch it
        weights = torch.tensor([1.0, 2.0]).to(model.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

training_args = TrainingArguments(
    output_dir="/content/gdrive/MyDrive/mental_health_prediction/unified_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Predictions on the Test Set
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Calculate Metrics
accuracy = accuracy_score(labels, preds)
recall = recall_score(labels, preds)
precision = precision_score(labels, preds)
f1 = f1_score(labels, preds)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Save the Model
save_path = "/content/gdrive/MyDrive/mental_health_prediction/final_unified_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model successfully saved to: {save_path}")
